In [ ]:
#检查有多少个病人, 考虑6:2:2分割

# 分析TRAIN38.mat中的prob_idx
import h5py
import numpy as np

# 加载TRAIN38.mat文件
f = h5py.File('TRAIN38.mat', 'r')
arrays = {}
for k, v in f.items():
    arrays[k] = np.array(v)
f.close()

# 提取prob_idx数据
prob_idx = arrays['prob_idx'].transpose()

# 分析prob_idx中的唯一值
unique_prob_idx = np.unique(prob_idx)
print(f"prob_idx中的唯一值: {unique_prob_idx}")
print(f"病人总数: {len(unique_prob_idx)}个")

# 检查每个病人的样本数量
for idx in unique_prob_idx:
    count = np.sum(prob_idx == idx)
    print(f"病人 {idx}: {count}个样本")

# 确认是否有38个病人
if 38 in unique_prob_idx or len(unique_prob_idx) == 38:
    print("数据集中确实有38个病人或prob_idx=38的数据")
else:
    print("数据集中没有38个病人，也没有prob_idx=38的数据")
    
# 检查数据集的基本情况
if 'data' in arrays and 'region' in arrays:
    data = arrays['data'].transpose()
    region = arrays['region'].transpose()
    print(f"\n数据集形状: {data.shape}")
    print(f"标签形状: {region.shape}")
    
    # 检查有多少个唯一的标签值
    unique_regions = np.unique(region)
    print(f"唯一标签值: {unique_regions}")
    print(f"标签类别数量: {len(unique_regions)}个")

In [ ]:
# data preprocessing
import tensorflow as tf
import h5py
import scipy.io
import numpy as np
from sklearn.model_selection import train_test_split
from tensorflow.keras import regularizers
from sklearn.preprocessing import StandardScaler

export_path = '/Your export path'
f = h5py.File('TRAIN38.mat','r')
arrays = {}
for k, v in f.items():
  arrays[k] = np.array(v)
f.close()
train_data = arrays['data'].transpose()
train_region = arrays['region'].transpose()
prob_idx = arrays['prob_idx'].transpose()
del arrays, f
curr_set = np.where(prob_idx != 38)[0]
set_data = train_data[curr_set,:]
set_region = train_region[curr_set,:]
print(set_data.shape)
curr_val = np.where(prob_idx == 38)[0]
val_data = train_data[curr_val,:]
val_label = train_region[curr_val,:]
print(val_data.shape)
del curr_set, curr_val, train_data, train_region
X_train1, X_train2, y_train1, y_train2 = train_test_split(set_data,set_region,test_size = 0.01, random_state = 42)
del set_data, set_region
scaler = StandardScaler()
scaler.fit(X_train1)
X_train1 = scaler.transform(X_train1)
val_data = scaler.transform(val_data)

In [ ]:
# 数据重组函数 - 将训练集和验证集合并并重新划分
def merge_and_shuffle_datasets(train_dir, val_dir, merged_dir):
    """
    合并train和val数据集，按标签组织，并对每个标签的体素进行随机打乱
   
    参数:
         train_dir: 原训练集目录路径
         val_dir: 原验证集目录路径
         merged_dir: 合并后的数据集保存路径
    """
    # 创建合并目录
    if not os.path.exists(merged_dir):
        os.makedirs(merged_dir)
       
    # 获取所有唯一标签
    train_files = glob.glob(os.path.join(train_dir, "label_*_count_*_voxels.npy"))
    val_files = glob.glob(os.path.join(val_dir, "val_label_*_count_*_voxels.npy"))
        
    label_ids = set()
    for f in train_files + val_files:
        # 从文件名提取标签ID
        label_id = int(os.path.basename(f).split('_')[1])
        label_ids.add(label_id)
        
    # 处理每个标签
    for label_id in tqdm(sorted(label_ids), desc="合并标签"):
        # 查找该标签的文件
        train_file = next((f for f in train_files if f"label_{label_id}_count" in os.path.basename(f)), None)
        val_file = next((f for f in val_files if f"val_label_{label_id}_count" in os.path.basename(f)), None)
            
        voxels = []
        total_count = 0
            
        # 加载训练集体素
        if train_file:
            train_voxels = np.load(train_file)
            voxels.append(train_voxels)
            total_count += len(train_voxels)
            
        # 加载验证集体素
        if val_file:
            val_voxels = np.load(val_file)
            voxels.append(val_voxels)
            total_count += len(val_voxels)
            
        if voxels:
            # 合并体素数据
            all_voxels = np.vstack(voxels)
                
            # 随机打乱体素
            indices = np.random.permutation(len(all_voxels))
            all_voxels = all_voxels[indices]
                
            # 保存合并后的体素数据
            output_file = os.path.join(merged_dir, f"label_{label_id}_count_{len(all_voxels)}_voxels.npy")
            np.save(output_file, all_voxels)
            print(f"标签 {label_id}: 合并了 {total_count} 个体素, 保存到 {output_file}")
        
    # 创建标签索引文件
    create_label_index_file(merged_dir)
        
    return merged_dir

def create_label_index_file(data_dir, output_file=None):
    """创建标签索引文件"""
    if output_file is None:
        output_file = os.path.join(data_dir, "label_index.txt")
        
    voxel_files = glob.glob(os.path.join(data_dir, "label_*_count_*_voxels.npy"))
        
    with open(output_file, 'w') as f:split_merged_dataset
        f.write("label_id,voxel_count,filename\n")
            
        for voxel_file in sorted(voxel_files, key=lambda x: int(os.path.basename(x).split('_')[1])):
            filename = os.path.basename(voxel_file)
            parts = filename.split('_')
            label_id = int(parts[1])
            count = int(parts[3])
                
            f.write(f"{label_id},{count},{filename}\n")
        
    print(f"标签索引文件已创建: {output_file}")

def split_merged_dataset(merged_dir, train_dir, test_dir, val_dir, split_ratio=[0.6, 0.2, 0.2]):
    """
    将合并后的数据集按比例拆分为训练、测试和验证集
        
    参数:
        merged_dir: 合并后的数据集目录
        train_dir: 拆分后的训练集保存目录
        test_dir: 拆分后的测试集保存目录
        val_dir: 拆分后的验证集保存目录
        split_ratio: 拆分比例，默认[0.6, 0.2, 0.2]
    """
    # 创建目标目录
    for directory in [train_dir, test_dir, val_dir]:
        if not os.path.exists(directory):
            os.makedirs(directory)
        
    # 获取所有体素文件
    voxel_files = glob.glob(os.path.join(merged_dir, "label_*_count_*_voxels.npy"))
        
    for voxel_file in tqdm(voxel_files, desc="拆分数据集"):
        # 从文件名提取信息
        filename = os.path.basename(voxel_file)
        parts = filename.split('_')
        label_id = int(parts[1])
            
        # 加载体素数据
        voxels = np.load(voxel_file)
        total_voxels = len(voxels)
            
        # 计算拆分点
        train_end = int(total_voxels * split_ratio[0])
        test_end = train_end + int(total_voxels * split_ratio[1])
            
        # 拆分数据
        train_voxels = voxels[:train_end]
        test_voxels = voxels[train_end:test_end]
        val_voxels = voxels[test_end:]
            
        # 保存拆分后的数据
        np.save(os.path.join(train_dir, f"label_{label_id}_count_{len(train_voxels)}_voxels.npy"), train_voxels)
        np.save(os.path.join(test_dir, f"label_{label_id}_count_{len(test_voxels)}_voxels.npy"), test_voxels)
        np.save(os.path.join(val_dir, f"label_{label_id}_count_{len(val_voxels)}_voxels.npy"), val_voxels)
            
        print(f"标签 {label_id}: 共 {total_voxels} 个体素，拆分为 {len(train_voxels)} 训练, {len(test_voxels)} 测试, {len(val_voxels)} 验证")
        
    # 为每个集合创建标签索引文件
    create_label_index_file(train_dir, os.path.join(train_dir, "label_index.txt"))
    create_label_index_file(test_dir, os.path.join(test_dir, "label_index.txt"))
    create_label_index_file(val_dir, os.path.join(val_dir, "label_index.txt"))